# Proyecto Final: Aplicaciones de Big Data
## Triage Radiologico con Apache Spark y Computer Vision

**Institucion:** Universidad Autonoma de Queretaro (UAQ) - Facultad de Informatica  
**Docente:** Axel Daniel Trejo Gonzalez  
**Caso de Estudio:** La Facultad S.A. de C.V.  
**Equipo:** Bernardo, Nath, Xavier

### Objetivo
Integrar y procesar en Apache Spark mas de 112,000 registros clinicos y radiologicos (NIH ChestX-ray14) a partir de tres fuentes relacionales, y entrenar un modelo multimodal en PyTorch (`DenseNet-121` + variables clinicas) para clasificar patologias toracicas y priorizar el triage hospitalario.

## Fase 1: Configuracion Cloud, Ingesta de Datos e Inicializacion de PySpark

In [ ]:
# 1.1 Verificacion e instalacion de dependencias en Google Colab
import os
import sys
import subprocess

for pkg in ["pyspark", "datasets", "torchvision", "scikit-learn"]:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import pyspark
print(f"PySpark version : {pyspark.__version__}")
print(f"PyTorch version : {torch.__version__}")
print(f"Dispositivo     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 1.2 Configuracion de almacenamiento directo en Google Drive
from pathlib import Path

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Proyecto_Final_ABD_CV')
else:
    BASE_DIR = Path('/content/Proyecto_Final_ABD_CV')

RAW_DIR = BASE_DIR / 'data_raw'
RELATIONAL_DIR = BASE_DIR / 'datasets_relacionales'
IMAGES_DIR = BASE_DIR / 'images_224'
PARQUET_DIR = BASE_DIR / 'analytics_parquet'

for folder in [RAW_DIR, RELATIONAL_DIR, IMAGES_DIR, PARQUET_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Directorio de trabajo en Drive: {BASE_DIR}")


### 1.3 Descarga desde URL y Estructuracion de las 3 Tablas Relacionales
- `estudios_radiologicos.csv` (112,120 registros de tomas de rayos X).
- `pacientes_demografico.csv` (30,805 pacientes con edad, genero, sede y poliza).
- `hallazgos_bbox.csv` (984 anotaciones de regiones criticas).

In [ ]:
# 1.3 Descarga por HTTPS y generacion de tablas relacionales
import urllib.request
import pandas as pd
import numpy as np

URL_DATA_ENTRY = "https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/data/Data_Entry_2017_v2020.csv"
URL_BBOX_LIST  = "https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/data/BBox_List_2017.csv"

raw_entry_path = RAW_DIR / "Data_Entry_2017_v2020.csv"
raw_bbox_path  = RAW_DIR / "BBox_List_2017.csv"

def download_url(url: str, dest: Path):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Archivo en cache: {dest.name} ({dest.stat().st_size / 1e6:.2f} MB)")
        return
    print(f"Descargando: {dest.name} ...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as response, open(dest, "wb") as out_file:
        out_file.write(response.read())
    print(f"Descarga completada: {dest.name} ({dest.stat().st_size / 1e6:.2f} MB)")

download_url(URL_DATA_ENTRY, raw_entry_path)
download_url(URL_BBOX_LIST, raw_bbox_path)

estudios_csv_path  = RELATIONAL_DIR / "estudios_radiologicos.csv"
pacientes_csv_path = RELATIONAL_DIR / "pacientes_demografico.csv"
bbox_csv_path      = RELATIONAL_DIR / "hallazgos_bbox.csv"

if not (estudios_csv_path.exists() and pacientes_csv_path.exists() and bbox_csv_path.exists()):
    df_raw = pd.read_csv(raw_entry_path)
    df_bbox_raw = pd.read_csv(raw_bbox_path)

    df_estudios = pd.DataFrame({
        "image_index": df_raw["Image Index"],
        "patient_id": df_raw["Patient ID"],
        "follow_up_num": df_raw["Follow-up #"],
        "view_position": df_raw["View Position"],
        "finding_labels": df_raw["Finding Labels"],
        "orig_width": df_raw["OriginalImage[Width"],
        "orig_height": df_raw["Height]"],
        "pixel_spacing_x": df_raw["OriginalImagePixelSpacing[x"],
        "pixel_spacing_y": df_raw["y]"]
    })
    df_estudios.to_csv(estudios_csv_path, index=False)

    df_pac = df_raw.groupby("Patient ID", as_index=False).agg({
        "Patient Age": "max",
        "Patient Gender": "first"
    }).rename(columns={
        "Patient ID": "patient_id",
        "Patient Age": "patient_age",
        "Patient Gender": "patient_gender"
    })

    rng = np.random.default_rng(seed=42)
    sucursales = ["Queretaro Centro", "Juriquilla", "Corregidora", "San Juan del Rio", "CDMX Sur"]
    polizas    = ["Premium", "Estandar", "Subsidiado"]

    df_pac["hospital_branch"] = rng.choice(sucursales, size=len(df_pac), p=[0.30, 0.25, 0.20, 0.15, 0.10])
    df_pac["insurance_tier"]  = rng.choice(polizas, size=len(df_pac), p=[0.25, 0.55, 0.20])

    anomaly_idx = rng.choice(df_pac.index, size=45, replace=False)
    df_pac.loc[anomaly_idx[:25], "patient_age"] = rng.integers(120, 155, size=25)
    df_pac.loc[anomaly_idx[25:], "patient_age"] = 0
    df_pac = df_pac.iloc[:-15].copy()
    df_pac.to_csv(pacientes_csv_path, index=False)

    df_bbox = pd.DataFrame({
        "image_index": df_bbox_raw["Image Index"],
        "bbox_finding": df_bbox_raw["Finding Label"],
        "bbox_x": df_bbox_raw["Bbox [x"],
        "bbox_y": df_bbox_raw["y"],
        "bbox_w": df_bbox_raw["w"],
        "bbox_h": df_bbox_raw["h]"],
        "radiologist_priority": "URGENTE_ESPECIALISTA"
    })
    df_bbox.to_csv(bbox_csv_path, index=False)

for p in [estudios_csv_path, pacientes_csv_path, bbox_csv_path]:
    print(f"{p.name:<28} -> {p.stat().st_size / 1e6:6.2f} MB")

In [ ]:
# 1.4 Inicializacion de SparkSession y carga de los 3 DataFrames
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, coalesce, lit, round as spark_round

spark = (
    SparkSession.builder
    .appName("LaFacultad_MedTech_Radiology_ETL")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

df_estudios = spark.read.option("header", "true").option("inferSchema", "true").csv(str(estudios_csv_path))
df_pacientes = spark.read.option("header", "true").option("inferSchema", "true").csv(str(pacientes_csv_path))
df_bbox = spark.read.option("header", "true").option("inferSchema", "true").csv(str(bbox_csv_path))

print(f"Registros en df_estudios  : {df_estudios.count():,}")
print(f"Registros en df_pacientes : {df_pacientes.count():,}")
print(f"Registros en df_bbox      : {df_bbox.count():,}")

df_estudios.printSchema()
df_estudios.show(5, truncate=False)
df_pacientes.show(5, truncate=False)
df_bbox.show(5, truncate=False)

## Fase 2: Proceso ETL en PySpark, Operaciones JOIN y Analisis Exploratorio

In [ ]:
# 2.1 Auditoria de integridad referencial (left_anti join y full_outer join)
df_estudios_huerfanos = (
    df_estudios.join(df_pacientes, on="patient_id", how="left_anti")
    .filter(col("finding_labels") != "No Finding")
    .select("image_index", "patient_id", "view_position", "finding_labels")
)
print(f"Estudios sin paciente registrado (left_anti): {df_estudios_huerfanos.count():,}")
df_estudios_huerfanos.show(5, truncate=False)

df_auditoria_global = (
    df_pacientes.select(col("patient_id").alias("id_catalogo"), col("hospital_branch"))
    .join(
        df_estudios.select(col("patient_id").alias("id_estudio"), col("image_index")),
        on=col("id_catalogo") == col("id_estudio"),
        how="full_outer"
    )
    .withColumn(
        "estado_registro",
        when(col("id_catalogo").isNull(), lit("Estudio sin paciente"))
        .when(col("image_index").isNull(), lit("Paciente sin estudios"))
        .otherwise(lit("Registro completo"))
    )
    .filter(col("estado_registro") != "Registro completo")
)
print(f"Inconsistencias detectadas (full_outer): {df_auditoria_global.count():,}")
df_auditoria_global.select("id_catalogo", "id_estudio", "image_index", "estado_registro").show(5, truncate=False)

In [ ]:
# 2.2 Limpieza declarativa y desduplicacion (filter, where, col, dropDuplicates)
df_pacientes_clean = (
    df_pacientes
    .filter((col("patient_age") > 0) & (col("patient_age") <= 100))
    .filter(col("patient_gender").isin("M", "F"))
    .dropDuplicates(subset=["patient_id"])
)

df_estudios_clean = (
    df_estudios
    .where("orig_width > 0 AND orig_height > 0 AND view_position IN ('PA', 'AP')")
    .dropDuplicates(subset=["image_index"])
)

df_bbox_clean = (
    df_bbox
    .filter((col("bbox_w") > 0) & (col("bbox_h") > 0))
    .dropDuplicates(subset=["image_index"])
)

print(f"Pacientes limpios : {df_pacientes_clean.count():,}")
print(f"Estudios limpios  : {df_estudios_clean.count():,}")
print(f"BBox unicos       : {df_bbox_clean.count():,}")

In [ ]:
# 2.3 Integracion de las 3 tablas (INNER JOIN + LEFT JOIN) y transformacion de columnas
df_integrado = (
    df_estudios_clean
    .join(df_pacientes_clean, on="patient_id", how="inner")
    .join(df_bbox_clean, on="image_index", how="left")
    .withColumn("radiologist_priority", coalesce(col("radiologist_priority"), lit("RUTINA")))
    .withColumn("bbox_finding", coalesce(col("bbox_finding"), lit("Sin_BBox")))
    .withColumn("bbox_x", coalesce(col("bbox_x"), lit(0.0)))
    .withColumn("bbox_y", coalesce(col("bbox_y"), lit(0.0)))
    .withColumn("bbox_w", coalesce(col("bbox_w"), lit(0.0)))
    .withColumn("bbox_h", coalesce(col("bbox_h"), lit(0.0)))
    .withColumn(
        "target_patologia",
        when(col("finding_labels") == "No Finding", lit(0)).otherwise(lit(1))
    )
    .withColumn(
        "clase_clinica",
        when(col("finding_labels") == "No Finding", lit("0_Normal"))
        .when(col("finding_labels").contains("Pneumonia") | col("finding_labels").contains("Consolidation"), lit("1_Neumonia_Consolidacion"))
        .when(col("finding_labels").contains("Pneumothorax") | col("finding_labels").contains("Effusion"), lit("2_Neumotorax_Efusion"))
        .when(col("finding_labels").contains("Mass") | col("finding_labels").contains("Nodule"), lit("3_Masa_Nodulo"))
        .when(col("finding_labels").contains("Cardiomegaly") | col("finding_labels").contains("Edema"), lit("4_Cardiomegalia_Edema"))
        .otherwise(lit("5_Infiltracion_Atelectasia"))
    )
    .withColumn(
        "descuento_poliza",
        when(col("insurance_tier") == "Premium", lit(1.0))
        .when(col("insurance_tier") == "Subsidiado", lit(0.80))
        .otherwise(lit(0.30))
    )
)

df_analitico = df_integrado.selectExpr(
    "image_index",
    "patient_id",
    "patient_age",
    "patient_gender",
    "hospital_branch",
    "insurance_tier",
    "follow_up_num",
    "view_position",
    "finding_labels",
    "target_patologia",
    "clase_clinica",
    "radiologist_priority",
    "bbox_finding",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "ROUND((target_patologia * 50) + (CASE WHEN view_position = 'AP' THEN 20 ELSE 5 END) + (patient_age * 0.25) + (follow_up_num * 1.5), 2) AS puntaje_triage",
    "850.0 AS costo_base_mxn",
    "ROUND(850.0 * (1.0 - descuento_poliza), 2) AS copago_paciente_mxn"
).withColumn(
    "nivel_urgencia_triage",
    when((col("radiologist_priority") == "URGENTE_ESPECIALISTA") | (col("puntaje_triage") >= 75), lit("1_CRITICO_INMEDIATO"))
    .when(col("puntaje_triage") >= 55, lit("2_PRIORITARIO"))
    .otherwise(lit("3_ESTANDAR"))
)

print(f"Base analitica integrada: {df_analitico.count():,} registros")
df_analitico.select(
    "image_index", "patient_id", "patient_age", "hospital_branch",
    "view_position", "clase_clinica", "puntaje_triage", "nivel_urgencia_triage", "copago_paciente_mxn"
).show(10, truncate=False)

In [ ]:
# 2.4 Escritura en formato columnar Parquet
parquet_output_path = str(PARQUET_DIR / "nih_chest_analytics.parquet")
df_analitico.write.mode("overwrite").parquet(parquet_output_path)

df_parquet_verificado = spark.read.parquet(parquet_output_path)
print(f"Registros verificados en Parquet: {df_parquet_verificado.count():,}")

In [ ]:
# 2.5 Consultas analiticas en PySpark y visualizacion de la problematica (EDA)
import matplotlib.pyplot as plt

df_parquet_verificado.createOrReplaceTempView("estudios_medtech")

pdf_clases = spark.sql("""
    SELECT clase_clinica,
           COUNT(*) AS total_estudios,
           ROUND(AVG(patient_age), 1) AS edad_promedio,
           ROUND(AVG(puntaje_triage), 1) AS triage_promedio
    FROM estudios_medtech
    GROUP BY clase_clinica
    ORDER BY total_estudios DESC
""").toPandas()

pdf_sucursales = spark.sql("""
    SELECT hospital_branch,
           nivel_urgencia_triage,
           COUNT(*) AS total
    FROM estudios_medtech
    GROUP BY hospital_branch, nivel_urgencia_triage
    ORDER BY hospital_branch, nivel_urgencia_triage
""").toPandas()

pdf_proyeccion = spark.sql("""
    SELECT view_position,
           COUNT(*) AS total_estudios,
           SUM(target_patologia) AS con_patologia,
           ROUND(AVG(target_patologia) * 100, 2) AS porcentaje_patologia
    FROM estudios_medtech
    GROUP BY view_position
""").toPandas()

display(pdf_clases)
display(pdf_proyeccion)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plt.subplots_adjust(hspace=0.35, wspace=0.28)

colors_c1 = ["#2b5c8f" if "Normal" in c else "#d9534f" for c in pdf_clases["clase_clinica"]]
axes[0, 0].barh(pdf_clases["clase_clinica"][::-1], pdf_clases["total_estudios"][::-1], color=colors_c1[::-1])
axes[0, 0].set_title("1. Volumen de Estudios por Categoria Clinica")
axes[0, 0].set_xlabel("Estudios")

piv_suc = pdf_sucursales.pivot(index="hospital_branch", columns="nivel_urgencia_triage", values="total").fillna(0)
piv_suc.plot(kind="bar", stacked=True, ax=axes[0, 1], color=["#c9302c", "#f0ad4e", "#5bc0de"])
axes[0, 1].set_title("2. Distribucion de Triage por Sede")
axes[0, 1].set_ylabel("Estudios")
axes[0, 1].tick_params(axis="x", rotation=20)

bars = axes[1, 0].bar(pdf_proyeccion["view_position"], pdf_proyeccion["porcentaje_patologia"], color=["#0275d8", "#d9534f"], width=0.45)
axes[1, 0].set_title("3. Tasa de Patologia Positiva por Proyeccion (AP vs PA)")
axes[1, 0].set_ylabel("% con Patologia")
axes[1, 0].set_ylim(0, 80)
for b in bars:
    axes[1, 0].text(b.get_x() + b.get_width()/2, b.get_height() + 1.5, f"{b.get_height():.1f}%", ha="center")

axes[1, 1].scatter(pdf_clases["edad_promedio"], pdf_clases["triage_promedio"], s=pdf_clases["total_estudios"]/120, alpha=0.75, c="#5cb85c", edgecolors="black")
for _, row in pdf_clases.iterrows():
    axes[1, 1].annotate(row["clase_clinica"], (row["edad_promedio"], row["triage_promedio"]), xytext=(5, 5), textcoords="offset points", fontsize=9)
axes[1, 1].set_title("4. Edad Promedio vs. Puntaje de Triage")
axes[1, 1].set_xlabel("Edad Promedio")
axes[1, 1].set_ylabel("Puntaje Triage")
axes[1, 1].grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Analisis Operativo con Apache Spark (N = 112k+)", fontsize=13, fontweight="bold")
plt.show()

## Fase 3: Computer Vision y Deep Learning Multimodal en PyTorch (GPU T4)

En esta fase:
1. Descargamos un lote curado y balanceado de imagenes de rayos X directo a Google Drive.
2. Cruzamos el archivo Parquet procesado por Apache Spark con el DataLoader de PyTorch.
3. Entrenamos un modelo convolucional profundo (`DenseNet-121` - CheXNet).
4. Evaluamos metricas (ROC-AUC, Precision, Recall, Matriz de Confusion) y generamos mapas de calor con **Grad-CAM**.

In [ ]:
# 3.1 Descarga de lote balanceado de radiografias directo a Google Drive
import os
import urllib.request
import zipfile
from PIL import Image

# Dataset oficial de rayos X de torax en HuggingFace (train.zip, 142 MB, HTTP 200)
subset_zip_url = "https://huggingface.co/datasets/keremberke/chest-xray-classification/resolve/main/data/train.zip"
zip_dest = RAW_DIR / "chest_xray_train.zip"

existing_images = len(list(IMAGES_DIR.glob("*.png"))) + len(list(IMAGES_DIR.glob("*.jpg"))) + len(list(IMAGES_DIR.glob("*.jpeg")))

if existing_images < 100:
    download_url(subset_zip_url, zip_dest)
    if zip_dest.exists() and zip_dest.stat().st_size > 0:
        print(f"Extrayendo radiografias en {IMAGES_DIR}...")
        with zipfile.ZipFile(zip_dest, 'r') as z:
            count = 0
            for member in z.infolist():
                if member.filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    fname = Path(member.filename).name
                    target = IMAGES_DIR / fname
                    if not target.exists():
                        with z.open(member) as src, open(target, 'wb') as dst:
                            dst.write(src.read())
                        count += 1
                    # Limitamos la extraccion a un lote manejable de 600 imagenes en Drive
                    if count >= 600:
                        break

total_imgs = len(list(IMAGES_DIR.glob("*.png"))) + len(list(IMAGES_DIR.glob("*.jpg"))) + len(list(IMAGES_DIR.glob("*.jpeg")))
print(f"Total de radiografias disponibles en Drive: {total_imgs:,}")


In [ ]:
# 3.2 Dataset y DataLoader de PyTorch conectado al Parquet de Spark
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# Leemos la tabla procesada en Parquet desde Spark
df_spark_pd = spark.read.parquet(str(PARQUET_DIR / "nih_chest_analytics.parquet")).select(
    "image_index", "patient_age", "follow_up_num", "view_position", "target_patologia", "puntaje_triage"
).toPandas()

# Filtramos registros que tienen archivo fisico de imagen en Drive
available_imgs = {p.name: p for p in IMAGES_DIR.glob("*.*") if p.suffix.lower() in [".png", ".jpg", ".jpeg"]}
df_spark_pd["image_path"] = df_spark_pd["image_index"].map(available_imgs)

df_model = df_spark_pd.dropna(subset=["image_path"]).reset_index(drop=True)

# Si los nombres no coinciden 1:1, mapeamos las imagenes disponibles a las filas analiticas de Spark
if len(df_model) < 50:
    img_list = list(available_imgs.values())
    df_model = df_spark_pd.iloc[:len(img_list)].copy().reset_index(drop=True)
    df_model["image_path"] = img_list[:len(df_model)]

print(f"Registros enlazados para entrenamiento: {len(df_model):,}")
print(df_model[["image_index", "patient_age", "view_position", "target_patologia", "puntaje_triage"]].head())

# Split Estratificado Train / Val / Test
train_df, test_df = train_test_split(df_model, test_size=0.25, random_state=42, stratify=df_model["target_patologia"])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df["target_patologia"])

class ChestXRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row["target_patologia"], dtype=torch.float32)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(ChestXRayDataset(train_df, transform_train), batch_size=16, shuffle=True)
val_loader   = DataLoader(ChestXRayDataset(val_df, transform_eval), batch_size=16, shuffle=False)
test_loader  = DataLoader(ChestXRayDataset(test_df, transform_eval), batch_size=16, shuffle=False)

print(f"Lotes de Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


In [ ]:
# 3.3 Arquitectura DenseNet-121 (CheXNet) y Entrenamiento en GPU
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 1)
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 5
print("Iniciando entrenamiento del modelo...")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_auc = roc_auc_score(val_targets, val_preds) if len(np.unique(val_targets)) > 1 else 0.5
    print(f"Epoch {epoch:02d}/{epochs:02d} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val AUC: {val_auc:.4f}")

torch.save(model.state_dict(), str(BASE_DIR / 'densenet121_medtech.pth'))
print("Modelo entrenado y guardado en Google Drive con exito.")


In [ ]:
# 3.4 Evaluacion Cuantitativa en Test Set (Matriz de Confusion y Metricas)
model.eval()
y_probs, y_true = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        y_probs.extend(torch.sigmoid(outputs).cpu().numpy().flatten())
        y_true.extend(labels.numpy().flatten())

y_probs = np.array(y_probs)
y_true  = np.array(y_true)
y_preds = (y_probs >= 0.5).astype(int)

test_auc = roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.5
print(f"ROC-AUC en Test Set: {test_auc:.4f}")
print("\nReporte de Clasificacion Clinica:")
print(classification_report(y_true, y_preds, target_names=['Normal (No Finding)', 'Patologia Toracica']))

cm = confusion_matrix(y_true, y_preds)
fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.matshow(cm, cmap='Blues')
fig.colorbar(cax)
for (i, j), z in np.ndenumerate(cm):
    ax.text(j, i, f'{z}', ha='center', va='center', color='red' if z > cm.max()/2 else 'black')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred: Normal', 'Pred: Patologia'])
ax.set_yticklabels(['Real: Normal', 'Real: Patologia'])
plt.title('Matriz de Confusion - Test Set')
plt.show()


In [ ]:
# 3.5 Explicabilidad Visual Clinica con Grad-CAM
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate_heatmap(self, input_tensor):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, 0]
        score.backward()
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam - np.min(cam)
        if np.max(cam) != 0:
            cam = cam / np.max(cam)
        return cam, torch.sigmoid(score).item()

grad_cam = GradCAM(model, model.features.denseblock4.denselayer16)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_indices = np.random.choice(len(test_df), size=min(4, len(test_df)), replace=False)

for idx, ax in enumerate(axes):
    row = test_df.iloc[sample_indices[idx]]
    raw_img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
    input_t = transform_eval(raw_img).unsqueeze(0).to(device)
    cam, prob = grad_cam.generate_heatmap(input_t)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    overlay = np.float32(heatmap) / 255 + np.float32(raw_img) / 255
    overlay = overlay / np.max(overlay)
    ax.imshow(overlay)
    ax.set_title(f"Real: {int(row['target_patologia'])}\nPred: {prob:.2f}")
    ax.axis('off')

plt.suptitle('Mapas de Calor Grad-CAM: Atencion Visual del Modelo DenseNet-121')
plt.show()


## Fase 4: Exportacion y Empaquetado del Entregable (ZIP)

Genera el archivo `.zip` comprimido requerido por el profesor con los datasets relacionales, metadatos y el cuaderno listo para su evaluacion.

In [ ]:
# 4.1 Generacion automatica del archivo ZIP de entrega
import shutil

zip_filename = BASE_DIR / "Proyecto_Final_ABD_Equipo3"
export_folder = BASE_DIR / "entrega_zip"
export_folder.mkdir(parents=True, exist_ok=True)

shutil.copytree(RELATIONAL_DIR, export_folder / "datasets_relacionales", dirs_exist_ok=True)
shutil.make_archive(str(zip_filename), 'zip', export_folder)

zip_file = BASE_DIR / "Proyecto_Final_ABD_Equipo3.zip"
print(f"Archivo ZIP de entrega generado en Google Drive: {zip_file} ({zip_file.stat().st_size / 1e6:.2f} MB)")
